In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns

In [6]:
# Paths and config
DATA_PATH = "../Data/owid-energy-data-clean.csv"
OUTPUT_PATH = "../results/"
FIG_PATH = "../figures/"
MODEL_NAME = "linear_regression"

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(FIG_PATH, exist_ok=True)

In [7]:
# Load data
df = pd.read_csv(DATA_PATH)
df = df.sort_values(["country", "year"])

In [8]:
df.columns

Index(['country', 'year', 'iso_code', 'coal_share_energy', 'gas_share_energy',
       'oil_share_energy', 'biofuel_share_energy', 'hydro_share_energy',
       'solar_share_energy', 'wind_share_energy', 'nuclear_share_energy',
       'energy_per_capita', 'log_gdp_per_capita', 'log_population'],
      dtype='object')

In [9]:
# Time-based train/test split (80/20)
train = df[df["year"] <= 2012]
test = df[df["year"] > 2012]

feature_cols = [
    "year", "log_population", "log_gdp_per_capita",
    "coal_share_energy", "gas_share_energy", "oil_share_energy",
    "biofuel_share_energy",
    "hydro_share_energy", "solar_share_energy", "wind_share_energy",
    "nuclear_share_energy",
]

X_train = train[feature_cols]
X_test = test[feature_cols]

y_train = np.log1p(train["energy_per_capita"]) 
y_test  = np.log1p(test["energy_per_capita"])

In [10]:
# Train Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)  

results_df = pd.DataFrame({
    "year": test["year"],
    "country": test["country"],
    "actual_energy_per_capita": y_test_actual,
    "predicted_energy_per_capita": y_pred_actual
})

results_file = os.path.join(OUTPUT_PATH, f"{MODEL_NAME}_predictions.csv")
results_df.to_csv(results_file, index=False)

In [11]:
# Evaluate
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
r2 = r2_score(y_test_actual, y_pred_actual)

metrics_df = pd.DataFrame({
    "model": ["Linear Regression"],
    "rmse": [rmse],
    "r2": [r2]
})

metrics_file = os.path.join(OUTPUT_PATH, f"{MODEL_NAME}_metrics.csv")
metrics_df.to_csv(metrics_file, index=False)

print(f"Saved predictions to {results_file}")
print(metrics_df)

Saved predictions to ../results/linear_regression_predictions.csv
               model          rmse        r2
0  Linear Regression  28539.499119  0.546361


In [12]:
# Actual vs Predicted Plot 
plt.figure(figsize=(10, 6))
# Scatter with transparency for overlapping points
plt.scatter(
    results_df["actual_energy_per_capita"], 
    results_df["predicted_energy_per_capita"], 
    alpha=0.4, 
    s=40, 
    color='steelblue'
)
# Regression line
sns.regplot(
    x="actual_energy_per_capita", 
    y="predicted_energy_per_capita", 
    data=results_df,
    scatter=False,
    color="black",
    line_kws={"linewidth": 2, "linestyle": "--"}
)
plt.xlabel("Actual Energy per Capita")
plt.ylabel("Predicted Energy per Capita")
plt.title("Actual vs Predicted Energy per Capita (Linear Regression)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, f"{MODEL_NAME}_actual_vs_pred.png"))
plt.close()

# Residual Plot
residuals = results_df["actual_energy_per_capita"] - results_df["predicted_energy_per_capita"]
results_df["residuals"] = residuals

plt.figure(figsize=(10, 6))
plt.scatter(
    results_df["predicted_energy_per_capita"], 
    results_df["residuals"], 
    alpha=0.4, 
    s=40, 
    color='tomato'
)
plt.axhline(0, color='black', linestyle='--', linewidth=2)
plt.xlabel("Predicted Energy per Capita")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted (Linear Regression)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, f"{MODEL_NAME}_residuals.png"))
plt.close()

print(f"Saved figures to {FIG_PATH}")

Saved figures to ../figures/
